# Week 3: Data Ingestion & Preparation Pipelines

**Week:** 3 of 5 | **Stack:** Python · Requests · DuckDB · Pandas · Parquet · Folium

---

## What You Will Learn This Week

| # | Topic | What It Covers |
|---|-------|----------------|
| 1 | **Data Ingestion** | Extracting raw data from a REST/OData API, landing it in Bronze |
| 2 | **Control Flow vs Data Flow** | Pipeline anatomy: ordering tasks vs transforming records |
| 3 | **Idempotency** | Why pipelines must be safe to re-run |
| 4 | **Data Profiling** | Statistically understanding raw data before touching it |
| 5 | **Data Quality & Validation** | Schema checks, null analysis, format rules, quarantine pattern |
| 6 | **Data Cleaning (ETL)** | Standardising formats, handling CBS "." nulls, type casting in Pandas |
| 7 | **ELT with DuckDB** | Transform after loading with SQL — same result, different philosophy |

---

## Architecture

```
CBS OData API  (83765NED — Kerncijfers wijken en buurten 2017)
     │
     ▼  Phase 1: extract_bronze()
┌─────────────┐
│  BRONZE     │  Raw JSON — source shape preserved, nothing changed
│  Landing    │
└─────────────┘
     │
     ▼  Phase 1B-1D: profile → validate → clean  (Pandas / ETL path)
     │
     ▼  Phase 2: ELT with DuckDB SQL
┌─────────────┐
│  SILVER     │  Cleaned Parquet — schema enforced, typed, deduplicated
│  Clean Zone │
└─────────────┘
     │
     ▼  Phase 3 Preview (full Gold modelling covered in Week 4)
┌─────────────┐
│  GOLD       │  Analytics Mart — aggregated, business-ready
│  (preview)  │
└─────────────┘
```

## About the Data: CBS 83765NED

**Kerncijfers wijken en buurten 2017** — Key figures for every Dutch municipality (gemeente), district (wijk), and neighbourhood (buurt).

- ~16,667 rows covering NL, provinces, municipalities, districts, and neighbourhoods
- **Real quality issue**: CBS uses `"."` to mark suppressed or unreliable values — not Python `None`
- **4 pages** of data via paginated OData API (5,000 rows per page)
- Free to access — no authentication required

In [ ]:
%pip install folium --quiet

In [ ]:
# =============================================================================
# CELL 1 — Imports & Environment Setup
# =============================================================================
import json
import time
import hashlib
import logging
import requests
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium                        # pip install folium  (see requirements.txt)
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('cbs_pipeline')

# ── Directory layout (Medallion on local disk) ────────────────────────────────
# WHY exist_ok=True?  This is IDEMPOTENCY in action: creating a directory that
# already exists is a no-op.  Re-running the setup cell never fails.
BASE_DIR       = Path.cwd()
WEEK3_DIR      = BASE_DIR / 'week3'
BRONZE_DIR     = WEEK3_DIR / 'bronze'
SILVER_DIR     = WEEK3_DIR / 'silver'
GOLD_DIR       = WEEK3_DIR / 'gold'
QUARANTINE_DIR = WEEK3_DIR / 'quarantine'

for d in [BRONZE_DIR, SILVER_DIR, GOLD_DIR, QUARANTINE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── CBS API configuration ─────────────────────────────────────────────────────
TABLE_ID  = '83765NED'
BASE_URL  = f'https://opendata.cbs.nl/ODataFeed/odata/{TABLE_ID}/TypedDataSet'
PAGE_SIZE = 5000

# ── English aliases for CBS column names ─────────────────────────────────────
# CBS uses coded column names (e.g. AantalInwoners_5).
# We map them to readable English aliases throughout the pipeline.
CBS_KEY_COLS = {
    'WijkenEnBuurten':                'region_code',
    'SoortRegio_2':                   'region_type',
    'Gemeentenaam_1':                 'municipality',
    'AantalInwoners_5':               'population',
    'Mannen_6':                       'males',
    'Vrouwen_7':                      'females',
    'k_0Tot15Jaar_8':                 'age_0_14',
    'k_65JaarOfOuder_12':             'age_65_plus',
    'Bevolkingsdichtheid_33':         'pop_density',
    'GemiddeldInkomenPerInwoner_66':  'avg_income_x1000',
    'HuishoudensMetEenLaagInkomen_72':'pct_low_income_hh',
}

PIPELINE_RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

log.info(f'Pipeline run ID : {PIPELINE_RUN_ID}')
log.info(f'Base directory  : {BASE_DIR}')
log.info(f'Directories ready (idempotent mkdir)')
log.info(f'Extraction URL  : {BASE_URL}')

---
## Phase 1 — Data Ingestion: The Bronze Layer

### What is the Bronze Layer?

The Bronze layer is the **raw landing zone**. It contains an exact, unmodified copy of data from the source system — a digital photograph of the source at a point in time.

**Golden rule: Never transform data in the Bronze layer.**

Why?
- If your transformation has a bug, you can always reprocess from Bronze without re-hitting the API.
- It gives you a complete audit trail — you can prove what the source sent you.
- Source systems change, rate-limit you, or disappear. Bronze is your insurance.

### Pagination Pattern

```
page 1: GET ?$top=5000&$skip=0      → 5000 records → continue
page 2: GET ?$top=5000&$skip=5000   → 5000 records → continue
page 3: GET ?$top=5000&$skip=10000  → 5000 records → continue
page 4: GET ?$top=5000&$skip=15000  → ~1400 records → partial → stop
```

---

### Control Flow vs Data Flow — Two Different Things

When engineers talk about a "pipeline", they use two distinct concepts:

| Term | What it controls | Example |
|------|-----------------|---------|
| **Control Flow** | The ORDER tasks run in, and the CONDITIONS for running | "Only run Silver if Bronze succeeded" |
| **Data Flow** | HOW individual records change as they pass through a stage | "Replace '.' with None, then cast to int" |

A DAG (Directed Acyclic Graph) manages **control flow**.
A transformation function (Pandas, SQL) manages **data flow**.
You need both to build a reliable pipeline.

---

### Idempotency — Run It Twice, Get the Same Result

> **Idempotent**: running a pipeline multiple times with the same source data produces the same output.

Why it matters:
- Pipelines fail and get retried. If re-running creates duplicates or overwrites good data, the retry makes things **worse**.
- `exist_ok=True` on `mkdir` is idempotent — creating an existing directory is a no-op.
- Writing to a **fixed filename** (not a timestamp-named file) is idempotent — the second run overwrites the first cleanly.
- **Appending** to a file is **NOT** idempotent — retrying doubles your data.

You will see idempotency applied in every code cell below.

In [ ]:
# =============================================================================
# PHASE 1 — Bronze Extraction (paginated ODataFeed)
# =============================================================================
def extract_bronze():
    log.info('=== PHASE 1: BRONZE EXTRACTION (paginated) ===')
    log.info(f'Source   : {BASE_URL}')
    log.info(f'Page size: {PAGE_SIZE:,} records')

    all_records, skip, page_num = [], 0, 0

    while True:
        page_num += 1
        params = {'$top': PAGE_SIZE, '$skip': skip, '$format': 'json'}
        try:
            response = requests.get(BASE_URL, params=params, timeout=60)
            response.raise_for_status()
        except requests.exceptions.Timeout:
            log.error('Request timed out — CBS API may be slow. Retry later.')
            return 0
        except requests.exceptions.HTTPError as e:
            log.error(f'HTTP error on page {page_num}: {e}')
            return 0

        records = response.json().get('value', [])
        if not records:
            log.info(f'Page {page_num}: empty — extraction complete.')
            break

        all_records.extend(records)
        log.info(f'Page {page_num}: fetched {len(records):,} records '
                 f'(total so far: {len(all_records):,})')

        if len(records) < PAGE_SIZE:
            break          # partial page = last page
        skip += PAGE_SIZE
        time.sleep(0.3)    # polite rate-limiting

    if not all_records:
        log.warning('Extraction returned 0 records.')
        return 0

    # Write consolidated JSON (idempotent: overwrites on re-run)
    file_path = BRONZE_DIR / 'all_data.json'
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(all_records, f, indent=2, ensure_ascii=False)

    # SHA-256 checksum for lineage
    checksum = hashlib.sha256(file_path.read_bytes()).hexdigest()

    meta = {
        'pipeline_run_id': PIPELINE_RUN_ID,
        'source_url':      BASE_URL,
        'table_id':        TABLE_ID,
        'extracted_at':    datetime.now().isoformat(),
        'page_count':      page_num,
        'record_count':    len(all_records),
        'sha256':          checksum,
    }
    with open(BRONZE_DIR / 'extraction_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

    log.info(f'Landed {len(all_records):,} records → {file_path.name}')
    log.info(f'SHA-256: {checksum[:20]}...')
    return len(all_records)


bronze_record_count = extract_bronze()

---
## Phase 1B — Ingestion Patterns

In production you choose an **ingestion pattern** based on source constraints, data volume, and reliability requirements.

### The four core patterns

| Pattern | How it works | When to use |
|---------|-------------|-------------|
| **Full load** | Fetch everything, overwrite Bronze every run | Small tables, no change tracking in source |
| **Incremental (cursor)** | Fetch only rows newer than `last_run_timestamp` | Source has a reliable `updated_at` column |
| **CDC (Change Data Capture)** | Stream INSERT/UPDATE/DELETE events from transaction log | High-volume OLTP (Kafka, Debezium) |
| **Paginated full load** | Full load split into pages, each page written as its own file | Large APIs with record caps, slow networks |

We already implemented **Full load** (single `all_data.json`). Below: **Paginated full load** as a comparison.

---

### Paginated full load — key properties

**Resumability** — if the job dies on page 3, pages 1 & 2 are already safe on disk:
```
bronze/pages/
    page_0001.json   ← done
    page_0002.json   ← done
    page_0003.json   ← job crashed here
    page_0004.json   ← not yet fetched
```

**Downstream query** — DuckDB reads all pages with a glob:
```sql
SELECT * FROM read_json_auto('data/bronze/cbs_neighborhoods/pages/*.json')
```

| | Single file | Per-page files |
|-|-------------|---------------|
| Code simplicity | ✅ Simple | ❌ More logic |
| Resumable on failure | ❌ Must restart fully | ✅ Skip completed pages |
| Memory usage | ❌ All in RAM | ✅ Write as you go |
| Downstream SQL | ✅ One path | ⚠️ Needs glob or concat |

In [ ]:
# =============================================================================
# INGESTION PATTERN: Per-page files with manifest (simplified)
# =============================================================================
# In production this would also include:
#   - Resume logic: skip pages whose file + checksum already match the manifest
#   - Checksum validation on reload: detect corrupted page files before processing
# Here we focus on the core concept: write-per-page + track in a manifest.

PAGES_DIR = BRONZE_DIR / 'pages'
PAGES_DIR.mkdir(exist_ok=True)
MANIFEST_PATH = PAGES_DIR / 'pages_manifest.json'


def _checksum(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _save_manifest(manifest: dict):
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')


def extract_bronze_paginated():
    """
    Fetch CBS data page-by-page.
    Each page is written to its own JSON file.
    A manifest records which pages are complete (enables resume in production).
    """
    manifest = {
        'pipeline_run_id': PIPELINE_RUN_ID,
        'source_url':      BASE_URL,
        'table_id':        TABLE_ID,
        'started_at':      datetime.now().isoformat(),
        'run_complete':    False,
        'pages':           {},   # page_num → {path, records, sha256}
    }
    _save_manifest(manifest)

    skip, page_num, total_records = 0, 0, 0

    while True:
        page_num += 1
        params = {'$top': PAGE_SIZE, '$skip': skip, '$format': 'json'}

        try:
            resp = requests.get(BASE_URL, params=params, timeout=60)
            resp.raise_for_status()
        except requests.RequestException as e:
            log.error(f'Page {page_num} failed: {e}')
            break

        records = resp.json().get('value', [])
        if not records:
            break

        # Write this page to its own file
        page_file = PAGES_DIR / f'page_{page_num:04d}.json'
        with open(page_file, 'w', encoding='utf-8') as f:
            json.dump(records, f, ensure_ascii=False)

        # Track in manifest
        manifest['pages'][str(page_num)] = {
            'path':    str(page_file),
            'records': len(records),
            'sha256':  _checksum(page_file),
        }
        _save_manifest(manifest)

        total_records += len(records)
        log.info(f'Page {page_num}: {len(records):,} records → {page_file.name}')

        if len(records) < PAGE_SIZE:
            break
        skip += PAGE_SIZE
        time.sleep(0.3)

    manifest['run_complete'] = True
    manifest['completed_at'] = datetime.now().isoformat()
    manifest['total_records'] = total_records
    _save_manifest(manifest)

    log.info(f'Paginated extraction complete: {page_num} pages, {total_records:,} records')
    log.info(f'Manifest: {MANIFEST_PATH}')
    return total_records


paginated_count = extract_bronze_paginated()
print(f'\nPer-page extraction: {paginated_count:,} records across {len(list(PAGES_DIR.glob("page_*.json")))} files')
print(f'Manifest written: {MANIFEST_PATH.name}')

---
## Phase 1C — Data Profiling: Know Your Data Before You Touch It

### Why Profile?

Professional data engineers **always** profile data before writing transformation logic.
Profiling answers:

- How many rows and columns does the source have?
- What is the actual data type of each column (not what the schema claims)?
- How many null values per column, and in what form do they appear?
- What is the cardinality (unique value count) of categorical columns?
- Are there outliers, impossible values, or suspicious distributions?

**Without profiling, you are writing cleaning code blindly.**

### CBS-Specific Issues to Look For

| Issue | What to look for | Why it matters |
|-------|-----------------|----------------|
| **"." nulls** | Numeric columns contain the string `"."` | Standard Python `notna()` won't catch them |
| **Mixed region granularities** | Same table has NL, PV, GM, WK, BU rows | Aggregations at the wrong level produce nonsense |
| **Trailing whitespace** | `"Amsterdam "` ≠ `"Amsterdam"` | GROUP BY and JOIN will produce duplicates |
| **String numerics** | Population is `"12345"` not `12345` | Arithmetic fails silently |
| **Region code padding** | `"GM0363    "` (padded with spaces to fixed width) | String comparisons fail |

In [ ]:
# =============================================================================
# PHASE 1C — Data Profiling
# =============================================================================
with open(BRONZE_DIR / 'all_data.json', 'r', encoding='utf-8') as f:
    raw_records = json.load(f)

df_raw = pd.DataFrame(raw_records)

print('=' * 60)
print('STEP 1: Basic shape')
print('=' * 60)
print(f'Rows     : {df_raw.shape[0]:,}')
print(f'Columns  : {df_raw.shape[1]}')

print('\n' + '=' * 60)
print('STEP 2: Key column inventory (our 11 target columns)')
print('=' * 60)
key_cols_present = [c for c in CBS_KEY_COLS if c in df_raw.columns]
col_inventory = pd.DataFrame({
    'cbs_column':   key_cols_present,
    'alias':        [CBS_KEY_COLS[c] for c in key_cols_present],
    'dtype':        [str(df_raw[c].dtype) for c in key_cols_present],
    'non_null':     [df_raw[c].notna().sum() for c in key_cols_present],
    'sample_value': [str(df_raw[c].dropna().iloc[0]) if df_raw[c].notna().any() else None
                     for c in key_cols_present],
})
display(col_inventory)

print('\n' + '=' * 60)
print('STEP 3: CBS "." null analysis — how much data is suppressed?')
print('=' * 60)
# CBS uses "." as a sentinel for statistically unreliable or privacy-suppressed values.
# This is NOT Python None/NaN — it SURVIVES pd.notna() checks!
print(f'  {"alias":<30} {"dot_count":>10} {"dot_pct":>9}')
print('  ' + '-' * 55)
for col, alias in CBS_KEY_COLS.items():
    if col in df_raw.columns:
        dot_count = (df_raw[col].astype(str).str.strip() == '.').sum()
        dot_pct   = dot_count / len(df_raw) * 100
        bar       = '█' * int(dot_pct / 2)
        print(f'  {alias:<30} {dot_count:>10,} {dot_pct:>8.1f}%  {bar}')

print('\n' + '=' * 60)
print('STEP 4: Region type distribution')
print('=' * 60)
if 'SoortRegio_2' in df_raw.columns:
    region_dist = df_raw['SoortRegio_2'].value_counts()
    for rtype, cnt in region_dist.items():
        print(f'  {rtype:<12} {cnt:>6,} rows')
    print('\n  Note: NL=national, PV=province, GM=municipality, WK=district, BU=neighbourhood')

print('\n' + '=' * 60)
print('STEP 5: Sample values')
print('=' * 60)
display(df_raw[list(CBS_KEY_COLS.keys())].head(5))

In [ ]:
# =============================================================================
# DATA QUALITY DASHBOARD — Visual summary of CBS data quality issues
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('CBS 83765NED — Data Quality Dashboard (Bronze)', fontsize=14, fontweight='bold')

# --- Panel 1: CBS "." suppression rate per key column ---
key_col_list = [c for c in CBS_KEY_COLS if c in df_raw.columns]
dot_rates    = [(df_raw[c].astype(str).str.strip() == '.').mean() * 100 for c in key_col_list]
aliases      = [CBS_KEY_COLS[c] for c in key_col_list]
bar_colors   = ['#e74c3c' if r > 10 else ('#f39c12' if r > 0 else '#2ecc71') for r in dot_rates]

axes[0].barh(aliases, dot_rates, color=bar_colors)
axes[0].set_xlabel('Suppression rate (%)')
axes[0].set_title('Missing Value Rate\n(CBS "." sentinel)', fontsize=11)
axes[0].axvline(x=10, color='red', linestyle='--', alpha=0.5, label='10% threshold')
axes[0].legend(fontsize=8)
for i, r in enumerate(dot_rates):
    if r > 0:
        axes[0].text(r + 0.3, i, f'{r:.0f}%', va='center', fontsize=8)

# --- Panel 2: Region type distribution ---
if 'SoortRegio_2' in df_raw.columns:
    region_counts = df_raw['SoortRegio_2'].value_counts()
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
    axes[1].bar(region_counts.index, region_counts.values, color=colors[:len(region_counts)])
    axes[1].set_xlabel('Region Type')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Dataset Composition\n(rows per geographic level)', fontsize=11)
    for i, (rt, cnt) in enumerate(region_counts.items()):
        axes[1].text(i, cnt + 30, f'{cnt:,}', ha='center', fontsize=9)

# --- Panel 3: Population histogram for Buurt rows ---
if 'AantalInwoners_5' in df_raw.columns and 'SoortRegio_2' in df_raw.columns:
    buurt_pop = df_raw[df_raw['SoortRegio_2'] == 'Buurt']['AantalInwoners_5']
    buurt_num = pd.to_numeric(buurt_pop.replace('.', None), errors='coerce').dropna()
    axes[2].hist(buurt_num, bins=50, color='#3498db', edgecolor='white', alpha=0.85)
    axes[2].set_xlabel('Population')
    axes[2].set_ylabel('Number of neighbourhoods')
    axes[2].set_title('Population Distribution\n(Buurt / neighbourhood level)', fontsize=11)
    med = buurt_num.median()
    if pd.notna(med):
        axes[2].axvline(med, color='red', linestyle='--', label=f'Median: {int(med):,}')
        axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()
print('✓ Dashboard rendered.')
print(f'  Observation: income columns have high suppression rates — CBS withholds data')
print(f'  for small populations to protect privacy.  This is expected, not an error.')

---
## Phase 1D — Data Quality: Contracts and Rules

### What is a Data Quality Check?

A **data quality check** is a programmatic assertion that a dataset meets a defined expectation.
Quality checks are run on Bronze data **before** any transformation.

### The Six Quality Dimensions (DAMA framework)

| Dimension | Definition | CBS Example |
|-----------|-----------|-------------|
| **Completeness** | All expected columns and rows are present | All 11 key columns exist |
| **Uniqueness** | No duplicate primary keys | One row per `WijkenEnBuurten` code |
| **Validity** | Values conform to the expected format/range | Region codes start with NL/PV/GM/WK/BU |
| **Consistency** | Related values are logically coherent | males + females ≈ total population |
| **Accuracy** | Values reflect the real-world truth | (hard to check without an oracle) |
| **Timeliness** | Data is current enough for its use case | (checked via extraction metadata) |

### The Quarantine Pattern

Instead of dropping bad records, we **quarantine** them:

```
df_raw  →  [quality rules]  →  df_clean      → Silver layer
                            ↓
                         df_quarantine  →  quarantine/ folder  (for review)
```

Why quarantine instead of delete?
- You maintain visibility into how much of your data is bad
- The source system owner needs to know about quality issues
- Regulators may ask you to prove every source record was accounted for
- A quarantined record can be **re-released** after the source fixes the issue

> **Note on null handling:** You will notice CBS `"."` values are handled in two places —
> in the Pandas cleaning step (Phase 1E) AND in the DuckDB SQL (Phase 2).
> This is **intentional**: the Pandas path demonstrates the **ETL** approach (fix before loading),
> and the SQL path demonstrates the **ELT** approach (fix inside the storage engine).
> In a real pipeline you would choose one, not both.

In [ ]:
# =============================================================================
# PHASE 1D — Data Quality Rules
# =============================================================================
quality_report = []


def run_quality_rule(name, severity, check_fn):
    passed, failing_idx, message = check_fn()
    status = '✓ PASS' if passed else ('⚠ WARN' if severity == 'WARNING' else '✗ FAIL')
    quality_report.append({
        'rule': name, 'severity': severity,
        'passed': passed, 'failing_count': len(failing_idx), 'message': message
    })
    print(f'  [{status}] {name}')
    if not passed:
        print(f'         → {message}')
    return failing_idx


# RULE 1: Schema completeness
def rule_schema_completeness():
    required = list(CBS_KEY_COLS.keys())
    missing  = [c for c in required if c not in df_raw.columns]
    if missing:
        return False, [], f'Missing columns: {missing}'
    return True, [], f'All {len(required)} required columns present'


# RULE 2: Primary key uniqueness
def rule_unique_region_code():
    dupes = df_raw[df_raw.duplicated(subset=['WijkenEnBuurten'], keep=False)]
    return len(dupes) == 0, dupes.index.tolist(), f'{len(dupes)} duplicate region codes found'


# RULE 3: Region code format
def rule_region_code_format():
    valid_prefixes = ('NL', 'PV', 'GM', 'WK', 'BU', 'LD')
    col     = df_raw['WijkenEnBuurten'].fillna('')
    invalid = df_raw[~col.str.startswith(valid_prefixes)]
    return len(invalid) == 0, invalid.index.tolist(),            f'{len(invalid)} codes do not start with a valid CBS prefix (NL/PV/GM/WK/BU/LD)'


# RULE 4: No completely empty shell rows
def rule_no_empty_shells():
    num_cols  = ['AantalInwoners_5', 'GemiddeldInkomenPerInwoner_66', 'Mannen_6', 'Vrouwen_7']
    available = [c for c in num_cols if c in df_raw.columns]

    def is_shell(row):
        return all(str(row[c]).strip() in ('.', 'None', '') for c in available)

    shell_mask = df_raw.apply(is_shell, axis=1)
    shells     = df_raw[shell_mask]
    return len(shells) == 0, shells.index.tolist(),            f'{len(shells)} shell rows (all numeric values suppressed)'


# RULE 5: Internal consistency — males + females ≈ population
def rule_gender_sum_consistent():
    needed = ['AantalInwoners_5', 'Mannen_6', 'Vrouwen_7']
    if not all(c in df_raw.columns for c in needed):
        return True, [], 'Columns not available — skipped'
    tmp = df_raw.copy()
    for c in needed:
        tmp[c] = pd.to_numeric(tmp[c].astype(str).str.strip().replace('.', None), errors='coerce')
    tmp = tmp.dropna(subset=needed)
    tmp['sum_m_f'] = tmp['Mannen_6'] + tmp['Vrouwen_7']
    tolerance      = tmp['AantalInwoners_5'] * 0.02
    inconsistent   = tmp[abs(tmp['sum_m_f'] - tmp['AantalInwoners_5']) > tolerance]
    return len(inconsistent) == 0, inconsistent.index.tolist(),            f'{len(inconsistent)} rows where M+F ≠ total population (±2% tolerance)'


# RULE 6: No missing region key
def rule_no_missing_title():
    col = 'WijkenEnBuurten'
    if col not in df_raw.columns:
        return True, [], f'{col} not present — skipped'
    bad = df_raw[df_raw[col].isna() | (df_raw[col].astype(str).str.strip() == '')]
    return len(bad) == 0, bad.index.tolist(), f'{len(bad)} rows with empty region key'


# =============================================================================
# RUN ALL RULES
# =============================================================================
print('=' * 60)
print('DATA QUALITY REPORT — CBS 83765NED')
print('=' * 60)

run_quality_rule('Schema Completeness',    'ERROR',   rule_schema_completeness)
run_quality_rule('Primary Key Uniqueness', 'ERROR',   rule_unique_region_code)
format_idx = run_quality_rule('Region Code Format',    'WARNING', rule_region_code_format)
shell_idx  = run_quality_rule('No Empty Shell Rows',   'WARNING', rule_no_empty_shells)
run_quality_rule('Gender Sum Consistency', 'WARNING', rule_gender_sum_consistent)
title_idx  = run_quality_rule('No Missing Region Key', 'ERROR',   rule_no_missing_title)

all_failing_idx = set(format_idx) | set(shell_idx) | set(title_idx)

print(f'\n{"─"*60}')
passed_rules = sum(1 for r in quality_report if r['passed'])
print(f'Rules passed : {passed_rules} / {len(quality_report)}')
print(f'Records to quarantine : {len(all_failing_idx):,}')
print(f'Records to pass through: {len(df_raw) - len(all_failing_idx):,}')

df_quarantine = df_raw.loc[list(all_failing_idx)].copy()
df_clean      = df_raw.drop(index=list(all_failing_idx)).reset_index(drop=True)

print(f'\n✓ Quality checks complete.')

---
## Phase 1E — Data Cleaning: The ETL Pattern

### The Core Cleaning Operations

Data cleaning is not about making data "pretty" — it is about making it **usable and trustworthy**.

| Pattern | What It Does | CBS Example |
|---------|-------------|-------------|
| **Null standardisation** | Replace all null representations with Python `None` | Replace `"."` with `None` |
| **Whitespace trimming** | Remove leading/trailing spaces from strings | `" Nederland "` → `"Nederland"` |
| **Type casting** | Convert strings to correct types (int, float, date) | `"12345"` → `12345` |
| **Deduplication** | Remove or flag duplicate rows | Keep first occurrence by region code |
| **Column pruning** | Keep only relevant columns | 109 CBS columns → 11 key columns |
| **Derived columns** | Add computed columns for downstream use | `avg_income_eur = avg_income_x1000 * 1000` |

### Why Quarantine Instead of Delete?

Silently dropping bad records is dangerous:
- You lose visibility into how much of your data is bad
- Analysts notice row count discrepancies and lose trust in the pipeline
- The source system owner needs to know about quality issues
- Regulators may ask you to prove every source record was accounted for

The quarantine file acts as a **Dead Letter Queue** — records that need human review.

### This is the ETL Pattern

**Note:** What you see in this cell IS the ETL pattern — we Transform data in Python **before** saving it.
In Phase 2, you will see the ELT alternative: load raw data into DuckDB first, then transform with SQL.

In [ ]:
# =============================================================================
# PHASE 1E — Data Cleaning (Python / ETL Pattern)
# =============================================================================

# STEP A: Quarantine bad records
if len(df_quarantine) > 0:
    df_quarantine = df_quarantine.copy()
    df_quarantine['quarantine_reason'] = 'failed_quality_rule'
    df_quarantine['quarantine_run_id'] = PIPELINE_RUN_ID
    df_quarantine['quarantine_at']     = datetime.now().isoformat()
    q_path = QUARANTINE_DIR / f'quarantine_{PIPELINE_RUN_ID}.json'
    df_quarantine.to_json(q_path, orient='records', indent=2, force_ascii=False)
    log.warning(f'{len(df_quarantine)} records written to quarantine → {q_path.name}')
else:
    log.info('No records quarantined — all passed quality rules.')

# STEP B: Column pruning + rename to English aliases
available_cols = {k: v for k, v in CBS_KEY_COLS.items() if k in df_clean.columns}
df_work = df_clean[list(available_cols.keys())].copy()
df_work.rename(columns=available_cols, inplace=True)
print(f'[1] Column pruning: {df_clean.shape[1]} → {df_work.shape[1]} columns')

# STEP C: Whitespace trimming
str_cols = df_work.select_dtypes(include='object').columns
for col in str_cols:
    df_work[col] = df_work[col].str.strip()
print(f'[2] Whitespace trimmed on {len(str_cols)} string columns')

# STEP D: Replace CBS "." null placeholder with Python None
numeric_col_names = ['population', 'males', 'females', 'age_0_14', 'age_65_plus',
                     'pop_density', 'avg_income_x1000', 'pct_low_income_hh']
dot_replacements = 0
for col in numeric_col_names:
    if col in df_work.columns:
        before = (df_work[col].astype(str) == '.').sum()
        df_work[col] = df_work[col].replace({'.': None})
        dot_replacements += before
print(f'[3] CBS "." nulls replaced: {dot_replacements:,} values set to None')

# STEP E: Type casting
int_cols   = ['population', 'males', 'females', 'age_0_14', 'age_65_plus', 'pop_density']
float_cols = ['avg_income_x1000', 'pct_low_income_hh']
for col in int_cols:
    if col in df_work.columns:
        df_work[col] = pd.to_numeric(df_work[col], errors='coerce').astype('Int64')
for col in float_cols:
    if col in df_work.columns:
        df_work[col] = pd.to_numeric(df_work[col], errors='coerce')
print(f'[4] Type casting: {len(int_cols)} → Int64, {len(float_cols)} → float64')

# STEP F: Derived columns
if 'avg_income_x1000' in df_work.columns:
    df_work['avg_income_eur'] = df_work['avg_income_x1000'] * 1000
if 'age_0_14' in df_work.columns and 'population' in df_work.columns:
    df_work['youth_ratio_pct'] = (df_work['age_0_14'] / df_work['population'] * 100).round(1)
df_work['pipeline_run_id'] = PIPELINE_RUN_ID
df_work['cleaned_at']      = datetime.now().isoformat()
print(f'[5] Derived columns added: avg_income_eur, youth_ratio_pct, pipeline metadata')

# STEP G: Deduplication
before_dedup = len(df_work)
df_work = df_work.drop_duplicates(subset=['region_code'], keep='first')
print(f'[6] Deduplication: removed {before_dedup - len(df_work)} duplicate region_code rows')

print(f'\n{"─"*50}')
print(f'Clean dataset: {len(df_work):,} rows × {df_work.shape[1]} columns')
print('\nSample (first 5 rows):')
display(df_work[['region_code', 'region_type', 'municipality', 'population',
                 'avg_income_eur', 'youth_ratio_pct']].head())

---
## Phase 2 — ELT Transformation: The Silver Layer

### What is the Silver Layer?

The Silver layer contains **cleaned, typed, and validated data** in an efficient columnar format (Parquet).
It is the single source of truth for all downstream analytics — every Gold table reads from Silver.

### ETL vs ELT — Two Philosophies

| | **ETL** | **ELT** |
|---|---------|---------|
| **Order** | Extract → **Transform** → Load | Extract → Load → **Transform** |
| **Where transform happens** | In Python/Spark *before* writing to disk | Inside the storage engine (SQL) *after* loading |
| **Best for** | Complex logic, ML feature engineering | Analytical workloads, SQL-fluent teams |
| **Our example** | Phase 1E: Pandas cleaning before saving | Phase 2: DuckDB SQL transforms raw JSON |
| **Tool** | Pandas, PySpark | DuckDB, Snowflake, BigQuery, dbt |

### Why We Use ELT Here (DuckDB)

DuckDB is an **in-process OLAP SQL engine**. It can:
- Read JSON, CSV, and Parquet files directly — no import step
- Execute ANSI SQL with analytical extensions (`QUALIFY`, window functions, `TRY_CAST`)
- Run entirely in memory — no server to start
- Write results to Parquet in one statement

The `COPY ... TO ... (FORMAT PARQUET)` pattern is the ELT idiom: raw JSON in → clean Parquet out.

### What This Step Does
1. Read raw JSON directly from `bronze/` using `read_json_auto()`
2. `TRIM()` all string fields
3. Replace CBS `"."` nulls using `NULLIF(TRIM(col), '.')`
4. Cast strings to correct types with `TRY_CAST` (returns `NULL` on failure — safe)
5. Add audit columns (`ingested_at`, `pipeline_run_id`)
6. Write result to Parquet in `silver/`

In [ ]:
# =============================================================================
# PHASE 2 — Silver Layer (ELT with DuckDB)
# =============================================================================
con = duckdb.connect(database=':memory:')
silver_parquet_path = SILVER_DIR / 'neighborhoods_clean.parquet'

silver_query = f"""
COPY (
    SELECT
        TRIM(WijkenEnBuurten)                                             AS region_code,
        TRIM(SoortRegio_2)                                                AS region_type,
        TRIM(Gemeentenaam_1)                                              AS municipality,

        TRY_CAST(NULLIF(TRIM(CAST(AantalInwoners_5       AS VARCHAR)), '.') AS INTEGER) AS population,
        TRY_CAST(NULLIF(TRIM(CAST(Mannen_6               AS VARCHAR)), '.') AS INTEGER) AS males,
        TRY_CAST(NULLIF(TRIM(CAST(Vrouwen_7              AS VARCHAR)), '.') AS INTEGER) AS females,
        TRY_CAST(NULLIF(TRIM(CAST(k_0Tot15Jaar_8         AS VARCHAR)), '.') AS INTEGER) AS age_0_14,
        TRY_CAST(NULLIF(TRIM(CAST(k_65JaarOfOuder_12     AS VARCHAR)), '.') AS INTEGER) AS age_65_plus,
        TRY_CAST(NULLIF(TRIM(CAST(Bevolkingsdichtheid_33 AS VARCHAR)), '.') AS INTEGER) AS pop_density,

        TRY_CAST(NULLIF(TRIM(CAST(GemiddeldInkomenPerInwoner_66   AS VARCHAR)), '.') AS FLOAT) AS avg_income_x1000,
        TRY_CAST(NULLIF(TRIM(CAST(HuishoudensMetEenLaagInkomen_72 AS VARCHAR)), '.') AS FLOAT) AS pct_low_income_hh,

        TRY_CAST(NULLIF(TRIM(CAST(GemiddeldInkomenPerInwoner_66 AS VARCHAR)), '.') AS FLOAT)
            * 1000                                                        AS avg_income_eur,

        CURRENT_TIMESTAMP  AS ingested_at,
        '{PIPELINE_RUN_ID}' AS pipeline_run_id

    FROM read_json_auto('{BRONZE_DIR / 'all_data.json'}')
    WHERE WijkenEnBuurten IS NOT NULL
      AND TRIM(WijkenEnBuurten) != ''
) TO '{silver_parquet_path}' (FORMAT PARQUET);
"""

log.info('=== PHASE 2: SILVER LAYER (ELT) ===')
con.execute(silver_query)

verify_df = con.execute(f"SELECT * FROM '{silver_parquet_path}' LIMIT 5").df()
row_count  = con.execute(f"SELECT COUNT(*) FROM '{silver_parquet_path}'").fetchone()[0]

log.info(f'Silver Parquet written: {row_count:,} rows → {silver_parquet_path.name}')
log.info(f'File size: {silver_parquet_path.stat().st_size / 1024:.1f} KB')

print('\nSilver layer sample (first 5 rows):')
display(verify_df[['region_code', 'region_type', 'municipality', 'population',
                   'avg_income_eur', 'ingested_at']])

---
## Phase 2B — ETL vs ELT: Side-by-Side Comparison

This cell runs the **same transformation** two ways so you can see the practical difference.

### ETL (Python/Pandas) — Transform Before Loading
- Write Python code to clean data → save result
- Better when logic is complex, requires ML models, or calls external APIs
- Harder to inspect intermediate state — you need to print DataFrames

### ELT (DuckDB SQL) — Load First, Transform with SQL
- Dump raw data to storage → run SQL to produce clean output
- Better for set-based operations, aggregations, joins — SQL is concise and declarative
- Easier to port to cloud warehouses (Snowflake, BigQuery, Redshift)

### Which Should You Use?
- **Use ETL when:** logic is procedural, requires Python libraries, involves row-by-row processing
- **Use ELT when:** transformations are SQL-expressible (filter, cast, join, aggregate)
- **Modern best practice:** ELT for analytical pipelines, ETL for ML feature pipelines

In [ ]:
# =============================================================================
# PHASE 2B — ETL vs ELT Side-by-Side (same result, two approaches)
# =============================================================================

print('=' * 60)
print('APPROACH 1: ETL (Python/Pandas) — Transform BEFORE loading')
print('=' * 60)

with open(BRONZE_DIR / 'all_data.json', 'r') as f:
    raw = json.load(f)

df_etl = pd.DataFrame(raw)
df_etl = df_etl[['WijkenEnBuurten', 'SoortRegio_2', 'Gemeentenaam_1',
                  'AantalInwoners_5', 'GemiddeldInkomenPerInwoner_66']].copy()
df_etl.columns = ['region_code', 'region_type', 'municipality', 'population', 'avg_income_x1000']
df_etl['region_code']      = df_etl['region_code'].str.strip()
df_etl['municipality']     = df_etl['municipality'].str.strip()
df_etl['population']       = pd.to_numeric(df_etl['population'], errors='coerce')
df_etl['avg_income_x1000'] = pd.to_numeric(df_etl['avg_income_x1000'], errors='coerce')
df_etl['avg_income_eur']   = df_etl['avg_income_x1000'] * 1000
df_etl = df_etl.dropna(subset=['region_code'])

etl_output = SILVER_DIR / 'etl_approach.parquet'
df_etl.to_parquet(etl_output, index=False)
print(f'ETL output: {len(df_etl):,} rows → {etl_output.name}')
print(f'Transform logic: Python Pandas — requires pandas knowledge')
display(df_etl.head(3))

print()
print('=' * 60)
print('APPROACH 2: ELT (DuckDB SQL) — Load first, Transform with SQL')
print('=' * 60)

elt_query = f"""
SELECT
    TRIM(WijkenEnBuurten)                                               AS region_code,
    TRIM(SoortRegio_2)                                                  AS region_type,
    TRIM(Gemeentenaam_1)                                                AS municipality,
    TRY_CAST(NULLIF(CAST(AantalInwoners_5              AS VARCHAR), '.') AS INTEGER) AS population,
    TRY_CAST(NULLIF(CAST(GemiddeldInkomenPerInwoner_66 AS VARCHAR), '.') AS FLOAT)   AS avg_income_x1000,
    TRY_CAST(NULLIF(CAST(GemiddeldInkomenPerInwoner_66 AS VARCHAR), '.') AS FLOAT)
        * 1000                                                          AS avg_income_eur
FROM read_json_auto('{BRONZE_DIR / 'all_data.json'}')
WHERE WijkenEnBuurten IS NOT NULL
"""
df_elt = con.execute(elt_query).df()
print(f'ELT output: {len(df_elt):,} rows (computed by DuckDB SQL engine)')
print(f'Transform logic: SQL — portable to any SQL warehouse')
display(df_elt.head(3))

print()
print('KEY DIFFERENCES:')
print(f'  ETL: ~10 lines of Pandas Python  |  ELT: 1 SQL statement')
print(f'  ETL: requires pandas  |  ELT: TRY_CAST handles errors in SQL')
print(f'  ETL: best for Python ML pipelines  |  ELT: best for analytical warehousing')
print(f'  Both produce identical results: {len(df_etl):,} = {len(df_elt):,} rows  ✓')

---
## Phase 3 — Gold Layer Preview (Week 4 Topic)

> **Gold is covered in depth in Week 4.** This is a 2-minute preview so you can see where the Silver data leads.

The Gold layer contains **aggregated, business-ready tables** that directly answer specific analytical questions.

**Key rules:**
- Gold always reads from **Silver**, never Bronze
- Each Gold table answers a **specific business question** — not a generic copy
- Gold can always be **rebuilt from Silver** if a business rule changes

Below: two quick queries on the Silver Parquet we just built.
In Week 4 you will build these as proper dimensional models with star schema, written to Parquet.

In [ ]:
# =============================================================================
# GOLD PREVIEW — 2 queries to show what Silver enables
# (No Parquet writes here — full Gold layer with output files is Week 4)
# =============================================================================
con_preview = duckdb.connect(':memory:')

# Query 1: Top 10 highest-income neighbourhoods — ONE per municipality
# We use ROW_NUMBER() OVER (PARTITION BY municipality) so that a single
# wealthy municipality (e.g. Wassenaar) cannot claim several spots.
# This gives a more geographically diverse and informative top-10.
df_gold_preview = con_preview.execute(f"""
SELECT
    region_code,
    municipality,
    population,
    ROUND(avg_income_eur, 0)    AS avg_income_eur,
    ROUND(pct_low_income_hh, 1) AS pct_low_income_hh
FROM (
    SELECT
        region_code,
        municipality,
        population,
        avg_income_eur,
        pct_low_income_hh,
        ROW_NUMBER() OVER (
            PARTITION BY municipality
            ORDER BY avg_income_eur DESC
        ) AS rn
    FROM '{silver_parquet_path}'
    WHERE region_type    = 'Buurt'
      AND population     >= 500
      AND avg_income_eur IS NOT NULL
)
WHERE rn = 1
ORDER BY avg_income_eur DESC
LIMIT 10
""").df()

print('Top 10 highest-income neighbourhoods — best neighbourhood per municipality:')
display(df_gold_preview)

# Query 2: Income statistics by region type
df_by_type = con_preview.execute(f"""
SELECT
    region_type,
    COUNT(*)                         AS region_count,
    ROUND(AVG(avg_income_eur), 0)    AS avg_income_eur,
    ROUND(MIN(avg_income_eur), 0)    AS min_income_eur,
    ROUND(MAX(avg_income_eur), 0)    AS max_income_eur
FROM '{silver_parquet_path}'
WHERE avg_income_eur IS NOT NULL
GROUP BY region_type
ORDER BY avg_income_eur DESC
""").df()

print('\nAverage income by region type:')
display(df_by_type)

### Why the same municipality can appear multiple times — and how we fix it

The CBS dataset is at the **Buurt (neighbourhood) level**. Every row is a single neighbourhood, and the `municipality` column is just metadata telling you which city it belongs to.

**The problem:** A simple `ORDER BY avg_income_eur DESC LIMIT 10` ranks individual neighbourhoods. A wealthy municipality like **Wassenaar** (the most affluent municipality in the Netherlands) has *several* top-ranked neighbourhoods, so it can dominate the chart and squeeze out other cities entirely.

| Without deduplication | With deduplication |
|---|---|
| Wassenaar (BU0503…) #1 | Wassenaar #1 |
| Wassenaar (BU0503…) #2 | Bloemendaal #2 |
| Wassenaar (BU0503…) #3 | Laren #3 |
| Amsterdam (BU0363…) #4 | Amsterdam #4 |
| … | … |

**The fix — `ROW_NUMBER() OVER (PARTITION BY municipality)`**

A **window function** assigns a rank *within each municipality group*, ordered by income descending. We then keep only `rn = 1` — the single best neighbourhood per municipality — before applying the global `LIMIT 10`.

```sql
ROW_NUMBER() OVER (
    PARTITION BY municipality   -- restart the counter for each city
    ORDER BY avg_income_eur DESC -- highest income = rank 1
) AS rn
```

This is a core pattern in dimensional modelling: window functions let you express "best/latest/first *within a group*" without a self-join.

> **Discussion:** Is one neighbourhood per municipality the right business rule here? What if two completely different cities happen to have the same name? (Hint: check `region_code` prefixes — `GM` codes are unique per municipality.)

In [ ]:
# =============================================================================
# VISUALISATION 1 — Top 10 Highest-Income Neighbourhoods (Bar Chart)
# =============================================================================
fig, ax = plt.subplots(figsize=(11, 6))

labels = df_gold_preview['municipality'] + ' (' + df_gold_preview['region_code'].str.strip() + ')'
values = df_gold_preview['avg_income_eur'] / 1000
norm_vals = (values - values.min()) / (values.max() - values.min() + 1e-9)
colors = plt.cm.RdYlGn(norm_vals)

bars = ax.barh(labels, values, color=colors)
ax.set_xlabel('Average Income (€ thousands per year)')
ax.set_title(
    'Top 10 Municipalities by Highest-Income Neighbourhood — Netherlands 2017\n'
    '(CBS 83765NED · Buurt level · min. 500 residents · one neighbourhood per municipality)',
    fontweight='bold'
)
ax.invert_yaxis()

for bar, val in zip(bars, df_gold_preview['avg_income_eur']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'€{val:,.0f}', va='center', fontsize=9)

ax.set_xlim(0, values.max() * 1.18)
plt.tight_layout()
plt.show()
print('Source: CBS StatLine · Table 83765NED · Kerncijfers wijken en buurten 2017')

In [ ]:
# =============================================================================
# VISUALISATION 2 — Municipality Income Map (Folium Choropleth)
# =============================================================================
# Step 1: build df_muni from Silver — no network needed for this part.
df_muni = con_preview.execute(f"""
    SELECT
        TRIM(region_code)        AS region_code,
        municipality,
        ROUND(avg_income_eur, 0) AS avg_income_eur
    FROM '{silver_parquet_path}'
    WHERE region_type    = 'Gemeente'
      AND avg_income_eur IS NOT NULL
""").df()
print(f'Municipality income data ready: {len(df_muni)} rows.')

# Step 2: try to fetch GeoJSON boundaries and render an interactive map.
# Several mirror URLs are tried in order — educational networks often block
# github.io CDN but allow raw.githubusercontent.com, or vice versa.
GEO_URLS = [
    'https://cartomap.github.io/nl/wgs84/gemeente_2023.geojson',
    'https://raw.githubusercontent.com/cartomap/nl/master/wgs84/gemeente_2023.geojson',
    'https://raw.githubusercontent.com/cartomap/nl/master/wgs84/gemeente_2021.geojson',
]

geo_json = None
for geo_url in GEO_URLS:
    try:
        print(f'  Trying {geo_url} …', end=' ')
        geo_resp = requests.get(geo_url, timeout=15)
        geo_resp.raise_for_status()
        geo_json = geo_resp.json()
        print('OK')
        break
    except Exception as _e:
        print(f'failed ({type(_e).__name__})')

if geo_json is not None:
    # ── Interactive choropleth map ──────────────────────────────────────────
    import folium
    m = folium.Map(location=[52.3, 5.3], zoom_start=7, tiles='CartoDB positron')
    folium.Choropleth(
        geo_data=geo_json,
        data=df_muni,
        columns=['region_code', 'avg_income_eur'],
        key_on='feature.properties.statcode',
        fill_color='YlOrRd',
        fill_opacity=0.8,
        line_opacity=0.2,
        legend_name='Average Income per Inhabitant (€/year)',
        nan_fill_color='lightgray',
        nan_fill_opacity=0.4,
    ).add_to(m)
    folium.LayerControl().add_to(m)
    display(m)
    print(f'✓ Map rendered — {len(df_muni)} municipalities plotted.')
    print('  Darker orange/red = higher average income.  Grey = data suppressed by CBS.')

else:
    # ── Static bar chart fallback (no network required) ────────────────────
    print('\nAll GeoJSON sources unavailable (network blocked). Showing bar chart instead.')
    top_muni = df_muni.nlargest(20, 'avg_income_eur').sort_values('avg_income_eur')
    norm = (top_muni['avg_income_eur'] - top_muni['avg_income_eur'].min())
    norm = norm / (norm.max() + 1e-9)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = plt.cm.YlOrRd(0.3 + norm * 0.7)
    bars = ax.barh(top_muni['municipality'], top_muni['avg_income_eur'] / 1000, color=colors)
    ax.set_xlabel('Average Income (€ thousands / year)')
    ax.set_title(
        'Top 20 Municipalities by Average Income — Netherlands 2017\n'
        '(CBS 83765NED · Gemeente level · map unavailable offline)',
        fontweight='bold'
    )
    for bar, val in zip(bars, top_muni['avg_income_eur']):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                f'€{val:,.0f}', va='center', fontsize=8)
    ax.set_xlim(0, top_muni['avg_income_eur'].max() / 1000 * 1.2)
    plt.tight_layout()
    plt.show()
    print('✓ Bar chart rendered. Re-run this cell with internet access to see the interactive map.')

---
## Week 3 — Formative Review

Work through these questions on your own. Discuss your answers with a classmate before looking anything up.

---

**Conceptual questions**

1. Why should the Bronze layer **never be modified** after landing?
   *(Think about what happens when your transformation code has a bug.)*

2. What is the difference between **ETL** and **ELT**?
   In which situation would you choose each? Give one real-world example of each.

3. Define **idempotency** in your own words.
   Is the following operation idempotent? Why or why not?
   ```python
   with open('output.csv', 'a') as f:   # 'a' = append mode
       df.to_csv(f, header=False)
   ```

4. A dataset has a column `age` with values `[25, 42, '.', 'N/A', None, 0]`.
   List **three** different null representations. Why is this a problem for type casting?

5. You have a quality rule that finds 800 records where `males + females ≠ population`.
   Should you drop these records, quarantine them, or fix them? Justify your answer.

---

**Scenario question**

Your pipeline runs every night at 02:00. On Tuesday night the CBS API is slow and the job crashes halfway through page 3 of 4. On Wednesday night the pipeline re-runs automatically.

- What does the Bronze layer look like after Tuesday's crash?
- What happens on Wednesday if you wrote a **single-file** extraction (`all_data.json`)?
- What happens on Wednesday if you wrote **per-page files** with a manifest?
- Which approach is more idempotent? Why?

---

*Week 4: Dimensional Modelling